In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from collections import defaultdict

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
pair_lengths = []

for row in dataset:
    encoded = tokenizer(
        row["sentence1"],
        row["sentence2"],
        truncation=False,
        add_special_tokens=True
    )
    pair_lengths.append(len(encoded["input_ids"]))

sorted_lengths = sorted(pair_lengths)
n = len(sorted_lengths)
short_threshold = sorted_lengths[n // 3]
medium_threshold = sorted_lengths[(2 * n) // 3]

def assign_bucket(length):
    if length <= short_threshold:
        return "short"
    elif length <= medium_threshold:
        return "medium"
    return "long"

buckets = [assign_bucket(length) for length in pair_lengths]
bucket_counts = {name: buckets.count(name) for name in ["short", "medium", "long"]}

print(f"Length thresholds -> short <= {short_threshold}, medium <= {medium_threshold}, long > {medium_threshold}")
print("Bucket counts:")
print(bucket_counts)
print(f"Min length: {min(pair_lengths)}, Max length: {max(pair_lengths)}, Avg length: {sum(pair_lengths)/len(pair_lengths):.2f}")

In [ ]:
batch_size = 32
labels = dataset["label"]
predictions = []
confidences = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

print("Overall evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
bucket_metrics = {}

for bucket_name in ["short", "medium", "long"]:
    idxs = [i for i, b in enumerate(buckets) if b == bucket_name]
    y_true = [labels[i] for i in idxs]
    y_pred = [predictions[i] for i in idxs]
    bucket_accuracy = accuracy_score(y_true, y_pred)
    bucket_precision, bucket_recall, bucket_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    bucket_cm = confusion_matrix(y_true, y_pred)
    bucket_metrics[bucket_name] = {
        "count": len(idxs),
        "accuracy": bucket_accuracy,
        "precision": bucket_precision,
        "recall": bucket_recall,
        "f1": bucket_f1,
        "confusion_matrix": bucket_cm.tolist()
    }

print("Bucket-wise metrics:")
for bucket_name in ["short", "medium", "long"]:
    m = bucket_metrics[bucket_name]
    print(f"bucket={bucket_name} count={m['count']} accuracy={m['accuracy']:.4f} precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1']:.4f}")
    print(f"confusion_matrix={m['confusion_matrix']}")

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
long_mistakes = []

for i, bucket_name in enumerate(buckets):
    if bucket_name == "long" and labels[i] != predictions[i]:
        long_mistakes.append({
            "idx": i,
            "length": pair_lengths[i],
            "true_label": labels[i],
            "pred_label": predictions[i],
            "confidence": confidences[i],
            "sentence1": dataset[i]["sentence1"],
            "sentence2": dataset[i]["sentence2"]
        })

long_mistakes = sorted(long_mistakes, key=lambda x: (-x["length"], -x["confidence"]))
num_examples_to_show = min(5, len(long_mistakes))

print(f"Representative mistakes from longest bucket: showing {num_examples_to_show} of {len(long_mistakes)}")
for item in long_mistakes[:num_examples_to_show]:
    print(f"Index: {item['idx']}")
    print(f"Bucket: long | tokenized_pair_length: {item['length']}")
    print(f"sentence1: {item['sentence1']}")
    print(f"sentence2: {item['sentence2']}")
    print(f"true label: {item['true_label']} ({label_map[item['true_label']]})")
    print(f"pred label: {item['pred_label']} ({label_map[item['pred_label']]})")
    print(f"confidence: {item['confidence']:.4f}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"overall_accuracy={accuracy:.4f}")
print(f"overall_precision={precision:.4f}")
print(f"overall_recall={recall:.4f}")
print(f"overall_f1={f1:.4f}")
print(f"length_threshold_short={short_threshold}")
print(f"length_threshold_medium={medium_threshold}")
for bucket_name in ["short", "medium", "long"]:
    m = bucket_metrics[bucket_name]
    print(f"bucket_{bucket_name}_count={m['count']}")
    print(f"bucket_{bucket_name}_accuracy={m['accuracy']:.4f}")
    print(f"bucket_{bucket_name}_precision={m['precision']:.4f}")
    print(f"bucket_{bucket_name}_recall={m['recall']:.4f}")
    print(f"bucket_{bucket_name}_f1={m['f1']:.4f}")